#### ============================================================
#### CARDIOVASCULAR DISEASE PREDICTION
#### Notebook 2: Machine Learning Modelling and Evaulation
#### Sudarshan Koirala and Srijan Pandey — ML Modelling and Evaulation
#### MSc IT with Data Analytics — UWS 2025-26
#### ============================================================

# Import Libraries

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import joblib

# XGBoost
from xgboost import XGBClassifier

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

print("All libraries imported successfully!")

All libraries imported successfully!


# Load Clean Dataset 

In [46]:
df = pd.read_csv('../data/cardio_clean.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())


Dataset loaded successfully!
Shape: (68455, 15)

Columns: ['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'bmi', 'bmi_category', 'age_category']

First 5 rows:
    age  gender  height  weight  ap_hi  ap_lo  cholesterol  gluc  smoke  alco  \
0  50.4       2     168    62.0    110     80            1     1      0     0   
1  55.4       1     156    85.0    140     90            3     1      0     0   
2  51.7       1     165    64.0    130     70            3     1      0     0   
3  48.3       2     169    82.0    150    100            1     1      0     0   
4  47.9       1     156    56.0    100     60            1     1      0     0   

   active  cardio        bmi bmi_category age_category  
0       1       0  21.967120       Normal        50-59  
1       1       1  34.927679        Obese        50-59  
2       0       1  23.507805       Normal        50-59  
3       1       1  28.710479   Overweight        40-49  


# Drop non-numeric and target columns from features

In [47]:
X = df.drop(['cardio', 'bmi_category', 'age_category'], axis=1)
y = df['cardio']

print("Features (X):")
print(list(X.columns))
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

Features (X):
['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'bmi']

Feature matrix shape: (68455, 12)
Target variable shape: (68455,)

Target distribution:
cardio
0    34581
1    33874
Name: count, dtype: int64


# Train Test Split (80/20)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # for reproducibility
    stratify=y          # maintain class balance in both sets
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"\nTesting target distribution:")
print(y_test.value_counts())


Training set size: (54764, 12)
Testing set size: (13691, 12)

Training target distribution:
cardio
0    27665
1    27099
Name: count, dtype: int64

Testing target distribution:
cardio
0    6916
1    6775
Name: count, dtype: int64


# Feature Scaling

In [26]:
scaler = StandardScaler()

# Fit on training data only, transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete!")
print(f"\nScaled training set mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Scaled training set std (should be ~1): {X_train_scaled.std():.4f}")

Feature scaling complete!

Scaled training set mean (should be ~0): -0.0000
Scaled training set std (should be ~1): 1.0000


# Logistic Regression (Baseline Model)

In [27]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# --- Key Metrics Summary ---
print("Logistic Regression — Key Metrics:")
lr_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score': [
        round(accuracy_score(y_test, lr_pred), 4),
        round(precision_score(y_test, lr_pred), 4),
        round(recall_score(y_test, lr_pred), 4),
        round(f1_score(y_test, lr_pred), 4),
        round(roc_auc_score(y_test, lr_prob), 4)
    ]
})
display(lr_metrics)

# --- Classification Report as Table ---
print("\nLogistic Regression — Classification Report:")
lr_report = classification_report(y_test, lr_pred,
                                   target_names=['No CVD', 'CVD'],
                                   output_dict=True)
lr_report_df = pd.DataFrame(lr_report).transpose().round(4)
display(lr_report_df)

Logistic Regression — Key Metrics:


,Metric,Score
0,Accuracy,0.7243
1,Precision,0.7530
2,Recall,0.6589
3,F1 Score,0.7028
4,ROC-AUC,0.7883



Logistic Regression — Classification Report:


,precision,recall,f1-score,support
No CVD,0.7023,0.7883,0.7428,6916.0000
CVD,0.7530,0.6589,0.7028,6775.0000
accuracy,0.7243,0.7243,0.7243,0.7243
macro avg,0.7277,0.7236,0.7228,13691.0000
weighted avg,0.7274,0.7243,0.7230,13691.0000


# Decision Tree Classifier

In [28]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_scaled, y_train)

# Predictions
dt_pred = dt_model.predict(X_test_scaled)
dt_prob = dt_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("Decision Tree — Key Metrics:")
dt_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score': [
        round(accuracy_score(y_test, dt_pred), 4),
        round(precision_score(y_test, dt_pred), 4),
        round(recall_score(y_test, dt_pred), 4),
        round(f1_score(y_test, dt_pred), 4),
        round(roc_auc_score(y_test, dt_prob), 4)
    ]
})
display(dt_metrics)

print("\nDecision Tree — Classification Report:")
dt_report = classification_report(y_test, dt_pred,
                                  target_names=['No CVD', 'CVD'],
                                  output_dict=True)
dt_report_df = pd.DataFrame(dt_report).transpose().round(4)
display(dt_report_df)

Decision Tree — Key Metrics:


,Metric,Score
0,Accuracy,0.6303
1,Precision,0.6267
2,Recall,0.6255
3,F1 Score,0.6261
4,ROC-AUC,0.6302



Decision Tree — Classification Report:


,precision,recall,f1-score,support
No CVD,0.6339,0.6350,0.6345,6916.0000
CVD,0.6267,0.6255,0.6261,6775.0000
accuracy,0.6303,0.6303,0.6303,0.6303
macro avg,0.6303,0.6303,0.6303,13691.0000
weighted avg,0.6303,0.6303,0.6303,13691.0000


# Random Forest Classifier

In [29]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("Random Forest — Key Metrics:")
rf_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score': [
        round(accuracy_score(y_test, rf_pred), 4),
        round(precision_score(y_test, rf_pred), 4),
        round(recall_score(y_test, rf_pred), 4),
        round(f1_score(y_test, rf_pred), 4),
        round(roc_auc_score(y_test, rf_prob), 4)
    ]
})
display(rf_metrics)

# --- Classification Report ---
print("\nRandom Forest — Classification Report:")
rf_report = classification_report(y_test, rf_pred,
                                  target_names=['No CVD', 'CVD'],
                                  output_dict=True)
rf_report_df = pd.DataFrame(rf_report).transpose().round(4)
display(rf_report_df)

Random Forest — Key Metrics:


,Metric,Score
0,Accuracy,0.7075
1,Precision,0.7119
2,Recall,0.6868
3,F1 Score,0.6991
4,ROC-AUC,0.7663



Random Forest — Classification Report:


,precision,recall,f1-score,support
No CVD,0.7034,0.7277,0.7154,6916.0000
CVD,0.7119,0.6868,0.6991,6775.0000
accuracy,0.7075,0.7075,0.7075,0.7075
macro avg,0.7077,0.7073,0.7072,13691.0000
weighted avg,0.7076,0.7075,0.7073,13691.0000


# XGBoost Classifier

In [30]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_scaled, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("XGBoost — Key Metrics:")
xgb_metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Score': [
        round(accuracy_score(y_test, xgb_pred), 4),
        round(precision_score(y_test, xgb_pred), 4),
        round(recall_score(y_test, xgb_pred), 4),
        round(f1_score(y_test, xgb_pred), 4),
        round(roc_auc_score(y_test, xgb_prob), 4)
    ]
})
display(xgb_metrics)

# --- Classification Report ---
print("\nXGBoost — Classification Report:")
xgb_report = classification_report(y_test, xgb_pred,
                                   target_names=['No CVD', 'CVD'],
                                   output_dict=True)
xgb_report_df = pd.DataFrame(xgb_report).transpose().round(4)
display(xgb_report_df)

XGBoost — Key Metrics:


,Metric,Score
0,Accuracy,0.7239
1,Precision,0.7436
2,Recall,0.6747
3,F1 Score,0.7075
4,ROC-AUC,0.7892



XGBoost — Classification Report:


,precision,recall,f1-score,support
No CVD,0.7078,0.7721,0.7386,6916.0000
CVD,0.7436,0.6747,0.7075,6775.0000
accuracy,0.7239,0.7239,0.7239,0.7239
macro avg,0.7257,0.7234,0.7230,13691.0000
weighted avg,0.7255,0.7239,0.7232,13691.0000


# Hyperparameter Tuning — GridSearchCV

In [31]:
# Define cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Logistic Regression Tuning

In [32]:
print("Tuning Logistic Regression...")
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [1000]
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    lr_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
lr_grid.fit(X_train_scaled, y_train)
print(f"Best params: {lr_grid.best_params_}")
print(f"Best CV ROC-AUC: {lr_grid.best_score_:.4f}")

Tuning Logistic Regression...
Fitting 10 folds for each of 10 candidates, totalling 100 fits
Best params: {'C': 0.1, 'max_iter': 1000, 'solver': 'lbfgs'}
Best CV ROC-AUC: 0.7921


# Decision Tree Tuning

In [33]:
print("Tuning Decision Tree...")
dt_params = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
dt_grid.fit(X_train_scaled, y_train)
print(f"Best params: {dt_grid.best_params_}")
print(f"Best CV ROC-AUC: {dt_grid.best_score_:.4f}")

Tuning Decision Tree...
Fitting 10 folds for each of 90 candidates, totalling 900 fits
Best params: {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best CV ROC-AUC: 0.7938


# Random Forest Tuning

In [34]:
print("Tuning Random Forest...")
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train_scaled, y_train)
print(f"Best params: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC: {rf_grid.best_score_:.4f}")

Tuning Random Forest...
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}
Best CV ROC-AUC: 0.8013


# XGBoost Tuning

In [35]:
print("Tuning XGBoost...")
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
    xgb_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train_scaled, y_train)
print(f"Best params: {xgb_grid.best_params_}")
print(f"Best CV ROC-AUC: {xgb_grid.best_score_:.4f}")

Tuning XGBoost...
Fitting 10 folds for each of 108 candidates, totalling 1080 fits
Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
Best CV ROC-AUC: 0.8022


In [49]:
# --- Tuning Results Summary ---
tuning_summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 
              'Random Forest', 'XGBoost'],
    'Best CV ROC-AUC': [
        round(lr_grid.best_score_, 4),
        round(dt_grid.best_score_, 4),
        round(rf_grid.best_score_, 4),
        round(xgb_grid.best_score_, 4)
    ],
    'Best Parameters': [
        str(lr_grid.best_params_),
        str(dt_grid.best_params_),
        str(rf_grid.best_params_),
        str(xgb_grid.best_params_)
    ]
})
print("Hyperparameter Tuning Summary:")
display(tuning_summary)

Hyperparameter Tuning Summary:


,Model,Best CV ROC-AUC,Best Parameters
0,Logistic Regression,0.7921,"{'C': 0.1, 'max_iter': 1000, 'solver': 'lbfgs'}"
1,Decision Tree,0.7938,"{'criterion': 'entropy', 'max_depth': 7, 'min_..."
2,Random Forest,0.8013,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_..."
3,XGBoost,0.8022,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [42]:
# --- Get best models ---
lr_best = lr_grid.best_estimator_
dt_best = dt_grid.best_estimator_
rf_best = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_

# --- Predictions from tuned models ---
models = {
    'Logistic Regression': (lr_best, X_test_scaled),
    'Decision Tree': (dt_best, X_test_scaled),
    'Random Forest': (rf_best, X_test_scaled),
    'XGBoost': (xgb_best, X_test_scaled)
}

results = []

for name, (model, X_test_data) in models.items():
    pred = model.predict(X_test_data)
    prob = model.predict_proba(X_test_data)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })

# --- Display Results Table ---
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print("Final Tuned Model Comparison:")
print(results_df.round(4))

# Save results to CSV
results_df.to_csv('../outputs/model_comparison.csv')
print("\nResults saved to outputs/model_comparison.csv")

Final Tuned Model Comparison:
                     Accuracy  Precision  Recall  F1 Score  ROC-AUC
Model                                                              
Logistic Regression    0.7241     0.7528  0.6586    0.7026   0.7883
Decision Tree          0.7268     0.7636  0.6489    0.7016   0.7897
Random Forest          0.7290     0.7602  0.6608    0.7070   0.7971
XGBoost                0.7303     0.7548  0.6739    0.7121   0.7982

Results saved to outputs/model_comparison.csv


# Save Trained Models

In [40]:
import joblib
import os

# Create models directory if not exists
os.makedirs('../models', exist_ok=True)

# Save all tuned models
joblib.dump(lr_best, '../models/logistic_regression.pkl')
joblib.dump(dt_best, '../models/decision_tree.pkl')
joblib.dump(rf_best, '../models/random_forest.pkl')
joblib.dump(xgb_best, '../models/xgboost.pkl')

# Save scaler as well — important for future predictions!
joblib.dump(scaler, '../models/scaler.pkl')

print("All models saved successfully!")
print("\nSaved files:")
for f in os.listdir('../models'):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f"  {f} ({size:.1f} KB)")

All models saved successfully!

Saved files:
  decision_tree.pkl (19.6 KB)
  logistic_regression.pkl (0.9 KB)
  random_forest.pkl (26180.4 KB)
  scaler.pkl (1.2 KB)
  xgboost.pkl (2404.7 KB)


## Cross validation analysis

In [43]:
# Cross Validation Scores for All Tuned Models
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = {}
tuned_models = {
    'Logistic Regression': lr_best,
    'Decision Tree': dt_best,
    'Random Forest': rf_best,
    'XGBoost': xgb_best
}

for name, model in tuned_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = {
        'Mean ROC-AUC': round(scores.mean(), 4),
        'Std Dev': round(scores.std(), 4),
        'Min': round(scores.min(), 4),
        'Max': round(scores.max(), 4)
    }

cv_df = pd.DataFrame(cv_results).transpose()
print("Cross Validation Results — All Tuned Models:")
display(cv_df)

Cross Validation Results — All Tuned Models:


,Mean ROC-AUC,Std Dev,Min,Max
Logistic Regression,0.7921,0.0043,0.7872,0.7992
Decision Tree,0.7938,0.0051,0.7873,0.8027
Random Forest,0.8013,0.0045,0.7956,0.8082
XGBoost,0.8022,0.0046,0.7954,0.8096


## Overfitting Analysis

In [44]:
overfit_results = []

for name, model in tuned_models.items():
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    overfit_results.append({
        'Model': name,
        'Training Accuracy': round(train_acc, 4),
        'Test Accuracy': round(test_acc, 4),
        'Difference': round(train_acc - test_acc, 4)
    })

overfit_df = pd.DataFrame(overfit_results).set_index('Model')
print("Overfitting Analysis — Training vs Test Accuracy:")
display(overfit_df)

Overfitting Analysis — Training vs Test Accuracy:


,Training Accuracy,Test Accuracy,Difference
Model,,,
Logistic Regression,0.7282,0.7241,0.0042
Decision Tree,0.7373,0.7268,0.0105
Random Forest,0.7550,0.7290,0.0260
XGBoost,0.7489,0.7303,0.0186
